In [2]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

#importing libraries
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
from flatten_json import flatten
from bson.objectid import ObjectId
import ipywidgets as widgets
from datetime import datetime

In [22]:
start_new = dt.datetime(2019,6,1)
end_new = dt.datetime(2019,6,20)
start_new = datetime.combine(start_new, datetime.min.time())
start_new = start_new - dt.timedelta(hours = 5, minutes = 30)
end_new = datetime.combine(end_new, datetime.min.time())
end_new = end_new - dt.timedelta(hours = 5, minutes = 30)
print(start_new,end_new,end_new-start_new)

2019-05-31 18:30:00 2019-06-19 18:30:00 19 days, 0:00:00


In [23]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
aw = []
c_users = cursor.superstars.users
for documents in c_users.find({'created_at': {'$lt':end_new,'$gte': start_new}},
                              {'created_at','login_details.last_request_at','device_id','sign_up_details.device'}): 
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
users = pd.DataFrame(dic_flattened)

users = users[['_id','created_at','login_details_last_request_at','sign_up_details_device_id']]
users.columns = ['user_id','created_at','last_request','device_id']

In [24]:
print(len(users))
users.head()

4303


,user_id,created_at,last_request,device_id
0,5cf172a967847c1b7cf75450,2019-05-31 18:30:01.033,2019-05-31 18:35:03.118,3b88d24116d7c2cb646679dab8943f3a
1,5cf172af81a4b91b766b9713,2019-05-31 18:30:07.372,2019-06-02 07:27:00.290,fc24f0ddcf704257d9dd1b869e0d0be6
2,5cf172b081a4b91b766b9727,2019-05-31 18:30:08.731,2019-05-31 18:35:03.507,bc078c0e2566e8139820d10b9eb33bf2
3,5cf172c59e0a5c4bc4991954,2019-05-31 18:30:29.182,2019-06-05 17:40:06.741,8e0e34c1b6c12336e649d6d46c7e8ae7
4,5cf1751a67847c1b7cf77f90,2019-05-31 18:40:26.094,2019-05-31 18:56:43.577,9ff55ec3da1ab7e2d6e72fa03d27e883


In [25]:
df = pd.read_csv('users_not_training_on_d0_old.csv')
df = df[['device_id','create_time']]
print(len(df))
df.head()

176


,device_id,create_time
0,0491d97e7af60354f4b86a22dd486d9e,1560947808337007
1,b97cc6225fc388b829c42b31905a5973,1560936441342007
2,f7dd687880bbd9560c411f76b25ee03c,1560906630280007
3,aa3ed0546f5e99b13df1d520b382d05f,1560877391748007
4,415d2ed10652ca1239bbbda42e3c04fc,1560797074579022


In [26]:
users = users[users['device_id'].isin(df['device_id'])]
users = pd.merge(users[['user_id','device_id']],df,on='device_id')
users = users[['user_id','create_time']]
users['create_time'] = (pd.to_datetime(users['create_time'], unit='us'))    
print(len(users))
users.head()

105


,user_id,create_time
0,5cf21c5c67847c1b7cfcff68,2019-06-14 04:51:39.682006
1,5cf36c7067847c1b7c07edd9,2019-06-13 08:45:03.950025
2,5cf3d765f588234bcaf258e1,2019-06-13 11:51:12.734008
3,5cf3f31b67847c1b7c0d1b3b,2019-06-09 15:27:53.769009
4,5cf4a5c281a4b91b7685c76e,2019-06-09 06:34:49.105006


In [27]:
team_cursor = cursor.superstars.teams
aw = []
for documents in team_cursor.find({'created_at': {'$lt': end_new, '$gte': start_new}},
                                  {"user":1, "created_at":1}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
teams = pd.DataFrame(dic_flattened)

teams = teams[teams["user"].isin(users["user_id"])]
teams = teams[["_id","user"]]
teams.columns = ["team_id", "user_id"]
len(teams)

105

In [28]:
teams.head()

,team_id,user_id
118,5cf21c5c67847c1b7cfcff70,5cf21c5c67847c1b7cfcff68
350,5cf36c7067847c1b7c07ede1,5cf36c7067847c1b7c07edd9
451,5cf3d765f588234bcaf258e9,5cf3d765f588234bcaf258e1
475,5cf3f31b67847c1b7c0d1b43,5cf3f31b67847c1b7c0d1b3b
551,5cf4a5c281a4b91b7685c776,5cf4a5c281a4b91b7685c76e


In [29]:
matches_new_cursor = cursor.superstars.matches
aw = []
for documents in matches_new_cursor.find({'created_at': {'$lt': end_new + dt.timedelta(hours = 24), '$gte': start_new}}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
matches = pd.DataFrame(dic_flattened)

matches = matches[matches["home_team_id"].isin(teams["team_id"])]
matches = matches[matches['status']==3]

matches = matches[["home_team_id","_id","start_time"]]
matches.columns = ["team_id","match_id","match_start_time"]
matches.head()

,team_id,match_id,match_start_time
2160,5cf21c5c67847c1b7cfcff70,5cf220720721db1fde9529dd,2019-06-01 06:51:30.780
2175,5cf21c5c67847c1b7cfcff70,5cf220f781a4b91b7670f32b,2019-06-01 06:53:43.049
2257,5cf21c5c67847c1b7cfcff70,5cf2234781a4b91b76710062,2019-06-01 07:03:35.407
2295,5cf21c5c67847c1b7cfcff70,5cf2250f0721db1fde954f49,2019-06-01 07:11:11.688
2318,5cf21c5c67847c1b7cfcff70,5cf2261b67847c1b7cfda05e,2019-06-01 07:15:39.472


In [30]:
team_matches = pd.merge(matches,teams,on='team_id')
user_matches = pd.merge(team_matches,users,on='user_id')
print(len(user_matches))
user_matches.head()

595


,team_id,match_id,match_start_time,user_id,create_time
0,5cf21c5c67847c1b7cfcff70,5cf220720721db1fde9529dd,2019-06-01 06:51:30.780,5cf21c5c67847c1b7cfcff68,2019-06-14 04:51:39.682006
1,5cf21c5c67847c1b7cfcff70,5cf220f781a4b91b7670f32b,2019-06-01 06:53:43.049,5cf21c5c67847c1b7cfcff68,2019-06-14 04:51:39.682006
2,5cf21c5c67847c1b7cfcff70,5cf2234781a4b91b76710062,2019-06-01 07:03:35.407,5cf21c5c67847c1b7cfcff68,2019-06-14 04:51:39.682006
3,5cf21c5c67847c1b7cfcff70,5cf2250f0721db1fde954f49,2019-06-01 07:11:11.688,5cf21c5c67847c1b7cfcff68,2019-06-14 04:51:39.682006
4,5cf21c5c67847c1b7cfcff70,5cf2261b67847c1b7cfda05e,2019-06-01 07:15:39.472,5cf21c5c67847c1b7cfcff68,2019-06-14 04:51:39.682006


In [31]:
user_matches = user_matches[user_matches['match_start_time']>user_matches['create_time']]
matches11 = user_matches[user_matches['match_id']!='5c90d9c35fa0db526b4b8435']
print(len(matches11))

303


In [32]:
matches_per_user = matches11.groupby('user_id').agg({'match_id':'count'})

In [33]:
matches_per_user_include = pd.merge(matches_per_user,users[['user_id']],on='user_id',how='right')

In [34]:
matches_per_user_include = matches_per_user_include.fillna(0)

In [35]:
matches_played = matches_per_user_include.groupby('match_id').agg({'user_id':'count'})

In [36]:
#matches_played.sort_values('team_id',ascending=False,inplace=True)
matches_played['percentage'] = (matches_played['user_id']/matches_played.user_id.sum())*100

In [37]:
matches_played

,user_id,percentage
match_id,,
0.0,63,60.000000
1.0,26,24.761905
2.0,6,5.714286
3.0,3,2.857143
4.0,1,0.952381
9.0,1,0.952381
18.0,2,1.904762
21.0,1,0.952381
52.0,1,0.952381
